# 10 - Adaptive controller experiments
Replays every val and test sequence through the AdaptivePark controller and the baselines, using the **cached YOLO11n and YOLO11s outputs** of notebook 07 (so every strategy sees exactly the same detections; no GPU needed).

**Modes** (spec section 12): *Reduced* (YOLO11n every `cadence` frames, frames in between skipped), *Standard* (YOLO11n), *Enhanced* (YOLO11s), *Forced Refresh* (a frame that would be skipped is run because the sequence starts, a slot reached the **maximum observation age**, or one slot's risk is **critical**). Hysteresis margin and minimum dwell (de-escalation only) stop rapid switching. Signals: difficulty D and global slot risk R (top-10 mean), weights from notebook 09, computed only from the current image and the last processed frame.
**Skipped frames** (spec section 13) show the last reliable state, marked stale, with its age; they never produce a vacant observation. Accuracy is measured on **every** frame, so a stale state that is wrong counts as an error.

**Strategies:** fixed YOLO11n, fixed YOLO11s, periodic YOLO11n (every 2 / 3 frames), scene-triggered without slot risk (D only), full AdaptivePark (D + slot risk + critical-slot and max-age refresh), and an **oracle** (YOLO11s exactly where it fixes a YOLO11n error; uses labels, an upper bound only).
**Compute cost** per frame in YOLO11n units: YOLO11s = 2.63 (7.9 / 3.0 ms per image, batched, Kaggle T4, notebook 06). **This is not a Jetson number**; call counts are reported so the Orin latencies can replace it later.

**Choice on val only**, rule fixed before test: target = recover at least half of the YOLO11n -> YOLO11s macro-F1 gain; among settings that reach it, the **cheapest**; if none does, the most accurate. A second operating point (cheapest setting with no loss against fixed YOLO11n) is also reported. Thresholds are searched as quantiles of the val D / risk distributions.

**Inputs (Add Input -> Your Work):** notebook 07 output (slot scores, cached detections of both models, slot registry), notebook 09 output (`artifacts/scene_risk/`: weights, scales, per-frame image features) and notebook 03 output (`artifacts/splits/`).
**Accelerator:** none (CPU). Internet ON only to clone the repo.
**How to run:** `SMOKE = True` + Run All (a few sequences). Then `SMOKE = False` + **Save Version -> Save & Run All**.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Settings and inputs
SMOKE = True                      # True: a few sequences per split. False: every val and test sequence
SMOKE_SEQUENCES = 8
LIGHT, HEAVY = "yolo11n", "yolo11s"
COST = {LIGHT: 1.0, HEAVY: round(7.9 / 3.0, 2)}   # Kaggle T4 batched ms/image ratio from notebook 06 (NOT a Jetson measurement)
GAIN_SHARE = 0.5                  # selection: recover at least this share of the n -> s val macro-F1 gain, as cheaply as possible
MAX_GAP_FACTOR = 3.0              # same sequences as notebooks 08 and 09
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

import os, json, time, csv, gzip, pickle, itertools
from collections import defaultdict, Counter
from datetime import datetime
from pathlib import Path
import numpy as np
from src.common.config import load_yaml, config_hash
from src.common.status import new_run_id, write_status
from src.training import yolo_train as yt
from src.evaluation import det_eval as de
from src.occupancy import association as asc, schema
from src.temporal import filters as tf
from src.scene_analysis import features as sf
from src.slot_risk import risk as rk
from src.controller import replay as rp
CCFG = load_yaml("configs/adaptive_controller.yaml")

mans = [p for p in yt.find_files(INPUT_ROOT, "split_manifest.csv") if p.parent.name == "splits"]
assert mans, "split_manifest.csv not found: attach notebook 03's output"
ROW_BY_ID = {int(r["image_id"]): r for r in de.read_manifest(mans[0])}
SPLIT_INFO = json.loads((mans[0].parent / "split_info.json").read_text())
occ = [p.parent for p in yt.find_files(INPUT_ROOT, "slot_scores_test.csv.gz") if p.parent.name == "occupancy"]
assert occ, "attach notebook 07's output (full run)"
OCC = occ[0]; st07 = json.loads((OCC / "status.json").read_text())
assert st07["state"] == "COMPLETE"
recs09 = [p for p in yt.find_files(INPUT_ROOT, "scene_risk_record.json") if p.parent.name == "scene_risk"]
assert recs09, "scene_risk_record.json not found: attach notebook 09's output (full run, not smoke)"
SCN = recs09[0].parent; R09 = json.loads(recs09[0].read_text())
assert not R09["smoke_test"] and R09["dataset_split_hash"] == SPLIT_INFO["manifest_hash"] and R09["notebook07_run_id"] == st07["run_id"], \
    "notebook 09 output does not match this split / notebook 07 run"
THR, ASSOC = {}, {}
for v in (LIGHT, HEAVY):
    rec = json.loads((OCC / "occupancy_record_{}.json".format(v)).read_text())
    assert not rec["smoke_test"] and rec["dataset_split_hash"] == SPLIT_INFO["manifest_hash"], v
    THR[v] = float(rec["chosen_on_val"]["conf_thr"]); ASSOC[v] = rec["chosen_on_val"]
W_D = np.array([R09["difficulty"]["weights"][f] for f in sf.FEATURES]); SCALES = R09["difficulty"]["scales"]
W_R = np.array([R09["slot_risk"]["weights"][c] for c in rk.COMPONENTS]); AGG = R09["slot_risk"]["aggregation"]
for f, x in zip(sf.FEATURES, W_D):
    assert abs(CCFG["difficulty"]["weights"][f] - x) < 1e-9, "configs/adaptive_controller.yaml D weights differ from notebook 09"
CTX = {"thr": THR, "wD": W_D, "wR": W_R, "agg": AGG, "age_ref_s": float(R09["slot_risk"]["max_age_s_for_A"]),
       "inst_frames": int(R09["instability_frames"]), "occ_scale": float(SCALES["occlusion"]), "light": LIGHT, "heavy": HEAVY}
SLOTS = {s["slot_id"]: s for s in json.loads((OCC / "slots.json").read_text())}
print("D weights", dict(zip([sf.SHORT[f] for f in sf.FEATURES], W_D)), "| R weights", dict(zip([rk.SHORT[c] for c in rk.COMPONENTS], W_R)), "| aggregation", AGG)
print("score thresholds", THR, "| cost per frame (YOLO11n units)", COST)
OUT = Path(WORK) / "artifacts" / "controller"
OUT.mkdir(parents=True, exist_ok=True)
SUF = "_smoke" if SMOKE else ""
CFG_HASH = config_hash({"controller": CCFG, "smoke": SMOKE, "smoke_sequences": SMOKE_SEQUENCES, "cost": COST, "gain_share": GAIN_SHARE,
                        "gap": MAX_GAP_FACTOR, "split": SPLIT_INFO["manifest_hash"], "nb07": st07["run_id"], "nb09": R09["run_id"]})
RUN_ID = new_run_id("ctl")
STATUS = OUT / "status.json"
write_status(STATUS, "10_adaptive_controller_experiments", RUN_ID, "RUNNING", CFG_HASH, "inputs loaded")

In [ ]:
# Per-frame inputs for both models (slot scores, detection features, slot occlusion), cached for restarts
def parse_ts(s):
    return datetime.fromisoformat(s.replace("Z", "")).timestamp()

def load_scores(split):
    cols = defaultdict(list)
    with gzip.open(str(OCC / "slot_scores_{}.csv.gz".format(split)), "rt") as f:
        for r in csv.DictReader(f):
            for k in ("image", "camera", "timestamp", "slot_id"):
                cols[k].append(r[k])
            cols["image_id"].append(int(r["image_id"])); cols["gt"].append(int(r["gt_occupied"])); cols["valid"].append(int(r["valid"]))
            for v in (LIGHT, HEAVY):
                cols[v].append(float(r["score_" + v]))
    return {k: np.array(x) for k, x in cols.items()}

def load_frame_features(split):
    with gzip.open(str(SCN / "frame_features_{}.csv.gz".format(split)), "rt") as f:
        return {r["image"]: r for r in csv.DictReader(f)}

PREP = {}
def build(split):
    o = load_scores(split); ff = load_frame_features(split)
    dets = {}
    for v in (LIGHT, HEAVY):
        z = np.load(str(OCC / "detections" / "{}_{}.npz".format(v, split)))
        dets[v] = {"dets": z["dets"], "starts": z["starts"], "row": {str(n): i for i, n in enumerate(z["names"])}}
    names, first, counts = np.unique(o["image"], return_index=True, return_counts=True)
    fr = np.argsort(first); names, first, counts = names[fr], first[fr], counts[fr]
    cam = o["camera"][first]; iid = o["image_id"][first]
    sec = np.array([parse_ts(t) for t in o["timestamp"][first]]); day = np.array([ROW_BY_ID[int(i)]["day"] for i in iid])
    sp = tf.split_sequences(cam, sec, day, MAX_GAP_FACTOR)
    cam_cols = defaultdict(dict)
    for c, s in zip(o["camera"], o["slot_id"]):
        cam_cols[c].setdefault(s, len(cam_cols[c]))
    seqs = []
    ids = sorted(set(sp["seq"].tolist()))
    if SMOKE:
        ids = [ids[i] for i in np.linspace(0, len(ids) - 1, min(SMOKE_SEQUENCES, len(ids))).astype(int)]
    for q in ids:
        fidx = np.nonzero(sp["seq"] == q)[0]; fidx = fidx[np.argsort(sec[fidx], kind="stable")]
        c = cam[fidx[0]]
        S = {"camera": c, "sec": sec[fidx], "n_cols": len(cam_cols[c]), "names": [str(names[f]) for f in fidx],
             "image_ids": iid[fidx], "cols": [], "valid": [], "gt": [], "slot_ids": [], "fixable": [], "B": [], "M": [],
             "out": {v: {"score": [], "detf": [], "occ": []} for v in (LIGHT, HEAVY)}}
        for f in fidx:
            a, b = first[f], first[f] + counts[f]
            sids = tuple(o["slot_id"][a:b]); valid = o["valid"][a:b].astype(bool); gt = o["gt"][a:b]
            S["cols"].append(np.array([cam_cols[c][s] for s in sids])); S["valid"].append(valid); S["gt"].append(gt); S["slot_ids"].append(sids)
            if sids not in PREP:
                PREP[sids] = asc.prepare_slots([SLOTS[s]["polygon"] for s in sids])
            for v in (LIGHT, HEAVY):
                D = dets[v]; k = D["row"][str(names[f])]; dd = D["dets"][D["starts"][k]:D["starts"][k + 1]]
                pairs = asc.image_pairs(PREP[sids], dd)
                S["out"][v]["score"].append(o[v][a:b]); S["out"][v]["detf"].append(sf.detection_features(dd, THR[v], len(sids)))
                S["out"][v]["occ"].append(rk.slot_occlusion(len(sids), pairs, THR[v], float(ASSOC[v]["overlap_thr"])))
            wn = (o[LIGHT][a:b] >= THR[LIGHT]) != gt.astype(bool); ws = (o[HEAVY][a:b] >= THR[HEAVY]) != gt.astype(bool)
            S["fixable"].append(int((wn & ~ws & valid).sum()))
            r9 = ff[str(names[f])]
            S["B"].append(float(r9["brightness"]))
            S["M"].append(np.nan if r9["motion_raw"] in ("", "None") else float(sf.scale(float(r9["motion_raw"]), SCALES["motion"])))
        S["B"], S["M"], S["fixable"] = np.array(S["B"]), np.array(S["M"]), np.array(S["fixable"])
        seqs.append(S)
    return seqs

CACHE = OUT / "inputs_cache{}.pkl".format(SUF)
SEQ = None
if CACHE.exists():
    c = pickle.load(open(str(CACHE), "rb"))
    if c.get("hash") == CFG_HASH:
        SEQ = c["seq"]; print("inputs: cached")
if SEQ is None:
    t0 = time.time()
    try:
        SEQ = {s: build(s) for s in ("val", "test")}
    except Exception as e:
        write_status(STATUS, "10_adaptive_controller_experiments", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500]); raise
    pickle.dump({"hash": CFG_HASH, "seq": SEQ}, open(str(CACHE), "wb"))
    print("inputs built in {:.0f} s".format(time.time() - t0))
for s in SEQ:
    print("{:<5} sequences {} | frames {} | fixable slots {}".format(s, len(SEQ[s]), sum(len(q["sec"]) for q in SEQ[s]), sum(int(q["fixable"].sum()) for q in SEQ[s])))

In [ ]:
# Baselines on VAL, signal distributions for the threshold grid
def run(split, strategy, cfg):
    outs = [rp.run_sequence(q, strategy, cfg, CTX) for q in SEQ[split]]
    return outs, rp.evaluate(SEQ[split], outs, COST, cfg.get("max_age_s"))

KEYS = ("accuracy", "macro_f1", "FV", "FO", "cost_per_frame", "skipped_share", "stale_share", "mean_served_age_min", "mode_switches_per_hour")
def show(name, m, mark=""):
    print("  {:<34} {:>8} {:>8} {:>6} {:>6} {:>6} {:>7} {:>7} {:>7} {:>8}{}".format(name, *[str(m[k]) for k in KEYS], mark))
HDR = "  {:<34} {:>8} {:>8} {:>6} {:>6} {:>6} {:>7} {:>7} {:>7} {:>8}".format("strategy", "accuracy", "macroF1", "FV", "FO", "cost", "skipped", "stale", "age_min", "switch/h")

BASE = {}
for name, strat, cfg in (("fixed YOLO11n", "fixed_n", {}), ("fixed YOLO11s", "fixed_s", {}), ("periodic YOLO11n c=2", "periodic_n", {"cadence": 2}),
                         ("periodic YOLO11n c=3", "periodic_n", {"cadence": 3}), ("oracle (labels; upper bound)", "oracle", {})):
    o_, m_ = run("val", strat, cfg); BASE[name] = {"strategy": strat, "cfg": cfg, "val": m_, "outs": o_ if strat == "fixed_n" else None}
print("VAL baselines"); print(HDR)
for n_, b in BASE.items():
    show(n_, b["val"])
# distributions of the signals as the controller would see them under Standard mode (fixed YOLO11n)
outs_n = BASE["fixed YOLO11n"]["outs"]
Dv = np.concatenate([o["D"] for o in outs_n]); Rgv = np.concatenate([o["Rg"] for o in outs_n]); Rmv = np.concatenate([o["Rmax"] for o in outs_n])
q = lambda x, p: None if p is None else float(np.nanquantile(x, p))
print("\nval quantiles | D:", [round(q(Dv, p), 3) for p in (0.2, 0.4, 0.6, 0.8, 0.9)], "| global risk:", [round(q(Rgv, p), 3) for p in (0.2, 0.4, 0.7, 0.8, 0.9)],
      "| max slot risk:", [round(q(Rmv, p), 3) for p in (0.95, 0.99)])
ACC_N, ACC_S = BASE["fixed YOLO11n"]["val"]["macro_f1"], BASE["fixed YOLO11s"]["val"]["macro_f1"]
TARGET = ACC_N + GAIN_SHARE * (ACC_S - ACC_N)
print("val macro-F1: fixed n {} | fixed s {} | target (n + {:.0%} of the gain) {:.4f}".format(ACC_N, ACC_S, GAIN_SHARE, TARGET))

In [ ]:
# Tune scene_only and adaptive on VAL: stage 1 thresholds, stage 2 hysteresis / dwell / cadence / max age
STAGE2_DEFAULT = {"margin": 0.0, "min_dwell": 1, "cadence": 2, "max_age_s": 1800}
def thresholds(dq, lq, rq, cq, use_risk):
    return {"d_hi": q(Dv, dq), "d_lo": q(Dv, lq), "r_hi": q(Rgv, rq) if use_risk else float("inf"), "r_lo": q(Rgv, lq) if use_risk else None,
            "r_crit": q(Rmv, cq) if use_risk else None, "use_risk": use_risk, "quantiles": {"d_hi": dq, "lo": lq, "r_hi": rq, "r_crit": cq}}

def pick(rows):
    ok = [r for r in rows if r["macro_f1"] >= TARGET]
    if ok:
        return min(ok, key=lambda r: (r["cost_per_frame"], -r["macro_f1"])), "cheapest reaching the target"
    return max(rows, key=lambda r: (r["macro_f1"], -r["cost_per_frame"])), "target not reached: most accurate"

def tune(strategy):
    use_risk = strategy == "adaptive"
    rows, t0 = [], time.time()
    grid1 = itertools.product((0.6, 0.7, 0.8, 0.9), (None, 0.2, 0.4), (0.7, 0.8, 0.9) if use_risk else (None,), (0.95, 0.99, None) if use_risk else (None,))
    for dq, lq, rq, cq in grid1:
        cfg = dict(thresholds(dq, lq, rq, cq, use_risk), **STAGE2_DEFAULT)
        _, m_ = run("val", strategy, cfg); rows.append({"stage": 1, "cfg": cfg, **{k: m_[k] for k in m_}})
    best1, why1 = pick(rows)
    for mg, dw, cad, age in itertools.product((0.0, 0.02, 0.05), (1, 2, 3), (2, 3), (900, 1800, 3600)):
        cfg = dict(best1["cfg"], margin=mg, min_dwell=dw, cadence=cad, max_age_s=age)
        _, m_ = run("val", strategy, cfg); rows.append({"stage": 2, "cfg": cfg, **{k: m_[k] for k in m_}})
    best, why = pick(rows)
    eff = [r for r in rows if r["macro_f1"] >= ACC_N]
    eff = min(eff, key=lambda r: (r["cost_per_frame"], -r["macro_f1"])) if eff else None
    print("{}: {} settings in {:.0f} s | chosen ({}): {}".format(strategy, len(rows), time.time() - t0, why,
          {k: (round(v, 4) if isinstance(v, float) else v) for k, v in best["cfg"].items()}))
    return rows, best, eff

TUNED = {}
try:
    for strat in ("scene_only", "adaptive"):
        TUNED[strat] = dict(zip(("rows", "best", "eff"), tune(strat)))
except Exception as e:
    write_status(STATUS, "10_adaptive_controller_experiments", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500]); raise
print("\nVAL (chosen settings)"); print(HDR)
for n_, b in BASE.items():
    show(n_, b["val"])
for strat, t in TUNED.items():
    show(strat + " (accuracy point)", t["best"], "   <- chosen")
    if t["eff"] is not None:
        show(strat + " (efficiency point)", t["eff"])
with open(str(OUT / "val_grid{}.csv".format(SUF)), "w", newline="") as f:
    w = csv.writer(f); w.writerow(["strategy", "stage", "cfg"] + list(KEYS))
    for strat, t in TUNED.items():
        for r in t["rows"]:
            w.writerow([strat, r["stage"], json.dumps({k: v for k, v in r["cfg"].items()}, default=str)] + [r[k] for k in KEYS])

In [ ]:
# TEST: every strategy once, at the frozen val choices; then per group
PLAN = [("fixed YOLO11n", "fixed_n", {}), ("fixed YOLO11s", "fixed_s", {}), ("periodic YOLO11n c=2", "periodic_n", {"cadence": 2}),
        ("periodic YOLO11n c=3", "periodic_n", {"cadence": 3}), ("scene-triggered, no slot risk", "scene_only", TUNED["scene_only"]["best"]["cfg"]),
        ("AdaptivePark (accuracy point)", "adaptive", TUNED["adaptive"]["best"]["cfg"])]
if TUNED["adaptive"]["eff"] is not None:
    PLAN.append(("AdaptivePark (efficiency point)", "adaptive", TUNED["adaptive"]["eff"]["cfg"]))
PLAN.append(("oracle (labels; upper bound)", "oracle", {}))
TEST, TEST_OUTS = {}, {}
for name, strat, cfg in PLAN:
    TEST_OUTS[name], TEST[name] = run("test", strat, cfg)
print("TEST (accuracy over every frame, skipped frames judged on the state they show; cost in YOLO11n units, Kaggle T4 ratio)")
print(HDR)
for name, m_ in TEST.items():
    show(name, m_)
a = TEST["AdaptivePark (accuracy point)"]
print("\nAdaptivePark accuracy point | calls", a["calls"], "| mode share", a["mode_share"], "| reasons", a["reasons"], "| max-age violations", a["max_age_violations"])

# per group: accuracy and cost of n, s and AdaptivePark on the frames of each group
def frame_rows(name):
    rows = []
    for q_, o in zip(SEQ["test"], TEST_OUTS[name]):
        for t in range(len(q_["sec"])):
            g = q_["gt"][t][q_["valid"][t]].astype(bool); p = o["pred"][t]; k = p >= 0
            r0 = ROW_BY_ID[int(q_["image_ids"][t])]
            rows.append({"groups": ["site:" + r0["site"], "climate:" + (r0.get("climate") or "unknown")], "right": int((p[k].astype(bool) == g[k]).sum()),
                         "n": int(k.sum()), "cost": COST[o["model"][t]] if o["model"][t] else 0.0})
    return rows
GROUP_ROWS = []
FRAMES = {n_: frame_rows(n_) for n_ in ("fixed YOLO11n", "fixed YOLO11s", "AdaptivePark (accuracy point)")}
groups = sorted({g for r in FRAMES["fixed YOLO11n"] for g in r["groups"]})
print("\n{:<26} {:>6} {:>9} {:>9} {:>9} {:>10}".format("group", "frames", "acc n", "acc s", "acc AP", "cost AP"))
for g in groups:
    row = {"group": g}
    for n_, rows in FRAMES.items():
        sel = [r for r in rows if g in r["groups"]]
        row[n_] = {"accuracy": round(sum(r["right"] for r in sel) / max(1, sum(r["n"] for r in sel)), 4), "cost_per_frame": round(sum(r["cost"] for r in sel) / max(1, len(sel)), 3), "frames": len(sel)}
    GROUP_ROWS.append(row)
    print("{:<26} {:>6} {:>9} {:>9} {:>9} {:>10}".format(g, row["fixed YOLO11n"]["frames"], row["fixed YOLO11n"]["accuracy"], row["fixed YOLO11s"]["accuracy"],
          row["AdaptivePark (accuracy point)"]["accuracy"], row["AdaptivePark (accuracy point)"]["cost_per_frame"]))

def clean(cfg):
    return {k: (round(v, 5) if isinstance(v, float) else v) for k, v in cfg.items()}
RECORD = {"run_id": RUN_ID, "config_hash": CFG_HASH, "smoke_test": SMOKE, "dataset_split_hash": SPLIT_INFO["manifest_hash"],
          "notebook07_run_id": st07["run_id"], "notebook09_run_id": R09["run_id"], "cost_units": COST,
          "cost_note": "YOLO11s/YOLO11n = 7.9/3.0 ms per image, batched, Kaggle T4 (notebook 06); NOT a Jetson measurement",
          "selection_rule": "val: macro-F1 >= n + {} x (s - n); cheapest such setting; else most accurate".format(GAIN_SHARE), "val_target_macro_f1": TARGET,
          "chosen": {s_: {"accuracy_point": clean(t["best"]["cfg"]), "efficiency_point": clean(t["eff"]["cfg"]) if t["eff"] else None} for s_, t in TUNED.items()},
          "val": {**{n_: b["val"] for n_, b in BASE.items()}, **{s_ + " accuracy point": t["best"] for s_, t in TUNED.items()}},
          "test": TEST, "groups_test": GROUP_ROWS,
          "semantics": "skipped frames show the last reliable state (stale, with age); never a fresh vacant; accuracy over every frame",
          "limitation": "frames 5 min (PKLot) / 30 min (CNRPark-EXT) apart; replay of cached detections; cost is a Kaggle T4 ratio"}
(OUT / "controller_record{}.json".format(SUF)).write_text(json.dumps(RECORD, indent=2, default=str))

In [ ]:
# Plots: val accuracy vs cost (every setting tried), test comparison, one sequence's mode timeline
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
a0 = ax[0]
for strat, mk in (("scene_only", "x"), ("adaptive", ".")):
    rows = TUNED[strat]["rows"]; a0.scatter([r["cost_per_frame"] for r in rows], [r["macro_f1"] for r in rows], marker=mk, s=14, alpha=0.5, label=strat + " settings")
for n_, b in BASE.items():
    a0.scatter([b["val"]["cost_per_frame"]], [b["val"]["macro_f1"]], marker="s", s=50, label=n_)
c = TUNED["adaptive"]["best"]; a0.scatter([c["cost_per_frame"]], [c["macro_f1"]], s=200, facecolors="none", edgecolors="k", label="chosen")
a0.axhline(TARGET, ls=":", color="0.4", lw=0.8)
a0.set(xlabel="cost per frame (YOLO11n units)", ylabel="macro-F1 (val)", title="VAL: every setting tried (dotted: target)"); a0.grid(alpha=0.3); a0.legend(fontsize=7)
a1 = ax[1]
names = list(TEST)
a1.scatter([TEST[n_]["cost_per_frame"] for n_ in names], [TEST[n_]["macro_f1"] for n_ in names])
for n_ in names:
    a1.annotate(n_, (TEST[n_]["cost_per_frame"], TEST[n_]["macro_f1"]), fontsize=7, xytext=(4, 2), textcoords="offset points")
a1.set(xlabel="cost per frame (YOLO11n units)", ylabel="macro-F1 (test)", title="TEST: strategies at the frozen val choices"); a1.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(str(OUT / "accuracy_vs_cost{}.png".format(SUF)), dpi=110); plt.show()

# mode timeline of the test sequence where AdaptivePark escalates most
outs = TEST_OUTS["AdaptivePark (accuracy point)"]
k = max(range(len(outs)), key=lambda i: sum(m == HEAVY for m in outs[i]["model"]))
q_, o = SEQ["test"][k], outs[k]
x = (q_["sec"] - q_["sec"][0]) / 60
fig, a = plt.subplots(figsize=(13, 3.6))
a.plot(x, o["D"], label="D"); a.plot(x, o["Rg"], label="global slot risk")
lvl = {"reduced": 0.05, "standard": 0.5, "enhanced": 0.95, "forced_refresh": 0.3}
a.step(x, [lvl[m] for m in o["mode"]], where="post", color="k", lw=1.5, label="mode (low=reduced, mid=standard, high=enhanced, 0.3=forced)")
sk = np.array([m is None for m in o["model"]]); a.scatter(x[sk], np.full(sk.sum(), 0.0), marker="|", color="0.5", label="skipped")
fx = q_["fixable"] > 0; a.scatter(x[fx], np.full(fx.sum(), 1.04), marker="v", color="r", label="YOLO11s would fix a slot")
a.set(xlabel="minutes", ylim=(-0.05, 1.1), title="test | {} | AdaptivePark".format(q_["camera"])); a.legend(fontsize=7, ncol=3, loc="center right")
plt.tight_layout(); plt.savefig(str(OUT / "mode_timeline{}.png".format(SUF)), dpi=110); plt.show()

In [ ]:
# Per-slot output records (spec schema) from the AdaptivePark replay: one executed frame and one skipped frame if any
EX = []
for q_, o in zip(SEQ["test"], TEST_OUTS["AdaptivePark (accuracy point)"]):
    want = [t for t in range(len(q_["sec"])) if o["model"][t] is not None][:1] + [t for t in range(len(q_["sec"])) if o["model"][t] is None][:1]
    for t in want:
        sids = np.array(q_["slot_ids"][t])[q_["valid"][t]]
        p, age = o["pred"][t], o["age_s"][t]
        for j in range(min(2, len(sids))):
            st = "unknown" if p[j] < 0 else ("occupied" if p[j] == 1 else "vacant")
            last_t = q_["sec"][t] - age[j] if not np.isnan(age[j]) else None
            last_i = int(np.argmin(np.abs(q_["sec"] - last_t))) if last_t is not None else None
            executed = o["model"][t] is not None
            sc = None
            if last_i is not None and o["model"][last_i] is not None and str(sids[j]) in q_["slot_ids"][last_i]:
                sc = round(float(q_["out"][o["model"][last_i]]["score"][last_i][list(q_["slot_ids"][last_i]).index(str(sids[j]))]), 4)
            EX.append(schema.slot_record(
                str(sids[j]), q_["camera"], q_["names"][t], datetime.fromtimestamp(q_["sec"][t]).isoformat(), st, sc,
                None if sc is None else float(rk.uncertainty(np.array([sc]), THR[o["model"][last_i]])[0]),
                [], q_["names"][last_i] if last_i is not None else None,
                datetime.fromtimestamp(last_t).isoformat() if last_t is not None else None,
                (t - last_i) if last_i is not None else None, None if np.isnan(age[j]) else float(age[j]),
                ("{} by {}".format(st, o["model"][t]) if executed else "frame skipped ({}): last reliable state shown, stale".format(o["mode"][t]))
                + ("; triggers: " + ", ".join(o["reasons"][t]) if o["reasons"][t] else ""),
                inference="executed" if executed else "skipped", controller_mode=o["mode"][t],
                transition_risk=None if np.isnan(o["Rg"][t]) else round(float(o["Rg"][t]), 4)))
    if len(EX) >= 4:
        break
skipped_ex = [r for r in EX if r["inference"] == "skipped"]
assert all(r["state"] != "vacant" or r["observation_age_frames"] > 0 for r in skipped_ex), "a skipped frame produced a fresh vacant"
(OUT / "example_slot_outputs{}.json".format(SUF)).write_text(json.dumps(EX, indent=1))
print(json.dumps(EX[:4], indent=1))

In [ ]:
# Finish: environment, status
from src.common.env_record import save_environment
save_environment(str(OUT / "environment.json"))
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch" and CACHE.exists():
    CACHE.unlink()                       # large; rebuilt in ~minutes from the inputs
write_status(STATUS, "10_adaptive_controller_experiments", RUN_ID, "COMPLETE", CFG_HASH, "smoke={}".format(SMOKE),
             extra={n_: {k: m_[k] for k in ("macro_f1", "accuracy", "cost_per_frame", "FV", "FO")} for n_, m_ in TEST.items()})
print("COMPLETE | saved to", OUT, sorted(p.name for p in OUT.iterdir()))

## What to send back
Download the notebook after **Save & Run All** (SMOKE = False) and send it, or paste: the inputs cell, the VAL baselines and quantiles, the tuning lines and VAL table, the TEST table and group table, the plots, and the example records.
`artifacts/controller/` (chosen settings, results) feeds notebooks 12-15, so **Save Version** with outputs.